## Cumulative responsibility of greenhouse gas emissions in Germany

In [1]:
import numpy as np
import pandas as pd

## Introduction

## Data Loading and Processing

### Data structure

In [2]:
raw_df = pd.read_csv(
        "../data:/85111-0005_en_flat.csv",
        sep=';',                    # Destatis commonly uses semicolons
        encoding='utf-8'            # Use 'cp1252' if utf-8 throws an error
    )

| Column Name | Description | Data Examples |
| :--- | :--- | :--- |
| **time** | Measurement Year | 2000, 2005, 2023... |
| **2_variable_attribute_label** | Emission Source (Broad Category) | Energy use, Road transport, Other processes |
| **3_variable_attribute_label** | Greenhouse Gas Type | Carbon dioxide ($CO_2$), Methane ($CH_4$), Total, etc. |
| **4_variable_attribute_label** | Industry Sector (WZ 2008) | Manufacture of paper, Fishing, etc. (Total 67 sectors) |
| **value_variable_label** | Emission Accounting Standard | Kyoto Protocol Total, All economic activities, Private households, etc. |
| **value** | Actual Emission Value | 4262142.558 (Numeric) |
| **value_unit** | Unit of Measurement | $t\ CO_2eq$ (tonnes of $CO_2$ equivalent) |

#### Macro data - national emission by year

In [3]:
# Filter data - only extract total emissions of year
total_em = raw_df[
    (raw_df['value_variable_label'] == 'Total greenhouse gas emissions acc. Kyoto Protocol') &
    (raw_df['3_variable_attribute_label'] == 'Total') # Using Total CO2e
].copy()

# Data type checking - change object data to numeric data for calculating
total_em['value'] = pd.to_numeric(total_em['value'], errors='coerce')

# Data accumulation - calculating sum emissions of all 3 sub-groups (Energy use, Road transport, Other processes)
total_em = total_em.groupby('time')['value'].sum().reset_index()

# Change the data column name
total_em = total_em[['time', 'value']].rename(columns={
    'time': 'year', 
    'value': 'emission'
}).reset_index(drop=True)

In [4]:
total_em.head()

,year,emission
0,2000,1.069205e+09
1,2001,1.087054e+09
2,2002,1.066397e+09
3,2003,1.070541e+09
4,2004,1.058412e+09


#### Micro data - sectoral emissions by year

1. Greenhouse gas emiss. of all economic activities (18 Sectors) +  Greenhouse gas emissions of private households =  Greenh. gas emissions of nationals (Total 19 Sectors)

In [5]:
# Filter data - only extract sectoral emissions
sector_em_raw = raw_df[
    (raw_df['value_variable_label'] == 'Greenhouse gas emissions') &
    (raw_df['3_variable_attribute_label'] == 'Total') &
    (raw_df['4_variable_attribute_code'].str.match(r'^WZ08-[A-Z]+(-[A-Z]+)*$', na=False)) # Not include all 67 sectors, only include higher economic sector classification (e.g., WZ08-A)
].copy()

# Convert value to numeric (crucial for correct summation)
sector_em_raw['value'] = pd.to_numeric(sector_em_raw['value'], errors='coerce')

# Rename the column of data
rename_dict = {
    "time": "year",
    "4_variable_attribute_label": "sector_name",
    "4_variable_attribute_code": "sector_code",
    "2_variable_attribute_label": "source",
    "value": "emission"
}
sector_em = sector_em_raw[list(rename_dict.keys())].rename(columns=rename_dict)

# Calculate the sum for whole sectors by sector_em_raw data
calculated_total = sector_em.groupby('year')['emission'].sum()

# Test the data with offical sum data for validating 
official_sector_total = raw_df[
    (raw_df['value_variable_label'] == 'Greenhouse gas emiss. of all economic activities') &
    (raw_df['3_variable_attribute_label'] == 'Total')
].copy()
official_sector_total['value'] = pd.to_numeric(official_sector_total['value'], errors='coerce') 
official_sector_total_series = official_sector_total.groupby('time')['value'].sum()

comparison = np.isclose(calculated_total, official_sector_total_series)
if comparison.all():
    print("Match")
else:
    mismatched_years = calculated_total.index[~comparison]
    print(f"Mismatch found in years: {list(mismatched_years)}")

Match


In [6]:
household_raw = raw_df[
    (raw_df['value_variable_label'] == 'Greenhouse gas emissions of private households') &
    (raw_df['3_variable_attribute_label'] == 'Total')
].copy()
household_df = household_raw.rename(columns={
    "time": "year",
    "2_variable_attribute_label": "source",
    "value": "emission"
})
household_df["sector_name"] = "Private Households"
household_df["sector_code"] = "HH"  # Dummy code for households
target_columns = ["year", "sector_name", "sector_code", "source", "emission"]
household_df = household_df[target_columns].copy()

# 5. Convert emission to numeric to avoid 'object' type issues
household_df["emission"] = pd.to_numeric(household_df["emission"], errors="coerce")
sector_em = pd.concat([sector_em, household_df], ignore_index=True)

In [7]:
sector_em.head()

,year,sector_name,sector_code,source,emission
0,2008,Mining and quarrying,WZ08-B,Other processes,5.226395e+06
1,2010,"Electricity, gas, steam, air conditioning supply",WZ08-D,Road transport,6.625116e+05
2,2002,Human health and social work activities,WZ08-Q,Energy use,6.876039e+06
3,2017,"Water supply,sewerage,waste management,remedia...",WZ08-E,Other processes,1.176431e+07
4,2010,Mining and quarrying,WZ08-B,Other processes,3.879621e+06


2. Sub-sectors (19 sub-sectors) of Manufacturing sector

In [8]:
# Filter data - only extract manufacturing sub-sector emissions (WZ08-10 to WZ08-33)
mfg_sub_list = ['WZ08-10-03', 'WZ08-13-04', 'WZ08-16', 'WZ08-17', 'WZ08-18', 'WZ08-19', 'WZ08-20', 'WZ08-21','WZ08-22', 'WZ08-23', 'WZ08-24', 'WZ08-25', 'WZ08-26', 'WZ08-27', 'WZ08-28', 'WZ08-29', 'WZ08-30', 'WZ08-31-02', 'WZ08-33']
mfg_sub_raw = raw_df[
    (raw_df['value_variable_label'] == 'Greenhouse gas emissions') &
    (raw_df['3_variable_attribute_label'] == 'Total') &
    (raw_df['4_variable_attribute_code'].isin(mfg_sub_list)) # Simple and precise
].copy()

# Convert value to numeric (crucial for correct summation)
mfg_sub_raw['value'] = pd.to_numeric(mfg_sub_raw['value'], errors='coerce')

# Rename the column of data
rename_dict = {
    "time": "year",
    "4_variable_attribute_label": "sector_name",
    "4_variable_attribute_code": "sector_code",
    "2_variable_attribute_label": "source",
    "value": "emission"
}
mfg_sub_em = mfg_sub_raw[list(rename_dict.keys())].rename(columns=rename_dict)

# Calculate the sum for manufacturing sub-sectors by mfg_sub_em data
calculated_mfg_total = mfg_sub_em.groupby('year')['emission'].sum()

# Extract official WZ08-C (Manufacturing parent) total for validating 
# Comparing against the parent sector 'WZ08-C' to ensure sub-sectors sum up correctly
official_mfg_parent = raw_df[
    (raw_df['value_variable_label'] == 'Greenhouse gas emissions') &
    (raw_df['3_variable_attribute_label'] == 'Total') &
    (raw_df['4_variable_attribute_code'] == 'WZ08-C')
].copy()

official_mfg_parent['value'] = pd.to_numeric(official_mfg_parent['value'], errors='coerce') 
official_mfg_parent_series = official_mfg_parent.groupby('time')['value'].sum()

# Test the data with official parent sector data for validating 
comparison = np.isclose(calculated_mfg_total, official_mfg_parent_series)
if comparison.all():
    print("Match")
else:
    mismatched_years = calculated_mfg_total.index[~comparison]
    print(f"Mismatch found in years: {list(mismatched_years)}")

Match


In [9]:
mfg_sub_em.head()

,year,sector_name,sector_code,source,emission
2881,2021,Manufacture of other transport equipment,WZ08-30,Energy use,3.995959e+05
2884,2006,Manufacture of coke and refined petroleum prod...,WZ08-19,Energy use,2.948588e+07
2907,2015,Manufacture of coke and refined petroleum prod...,WZ08-19,Road transport,6.238636e+04
2993,2017,"Manuf. of motor vehicles, trailers, semi-trailers",WZ08-29,Road transport,8.055341e+05
2994,2020,Manufacture of machinery and equipment n.e.c.,WZ08-28,Road transport,5.395990e+05


#### Global greenhouse gas emissions data

Mega ton -> ton 

In [10]:
edgar_df = pd.read_csv('../data:/EDGAR_2025_GHG.csv')

# Reshape the data (Wide to Long)
em_id_cols = ['EDGAR Country Code', 'Country']
em_year_cols = [col for col in edgar_df.columns if col not in em_id_cols]

global_em = edgar_df.melt(
    id_vars=em_id_cols, 
    var_name='year', 
    value_name='emission_Mt'
)

# Renaming
global_em = global_em.rename(columns={
    'EDGAR Country Code': 'country_code',
    'Country': 'country_name_em'
})

# Numeric conversion
global_em['year'] = pd.to_numeric(global_em['year'])
global_em['emission'] = pd.to_numeric(global_em['emission_Mt'], errors='coerce')
global_em['emission'] = global_em['emission_Mt'] * 10**6 # Mega ton to ton
global_em = global_em.drop(columns=['emission_Mt'])

In [11]:
global_em.head()

,country_code,country_name_em,year,emission
0,ABW,Aruba,1970,3.696956e+04
1,AFG,Afghanistan,1970,1.529996e+07
2,AGO,Angola,1970,1.883435e+07
3,AIA,Anguilla,1970,3.397521e+03
4,AIR,International Aviation,1970,1.724780e+08


#### Global population data

In [ ]:
pop_df = pd.read_csv('../data:/API_SP.POP.TOTL_DS2_en_csv_v2_127039.csv')

# Reshape (Wide to Long)
pop_id_cols = ['Country Name', 'Country Code']
pop_year_cols = [col for col in pop_df.columns if col.isdigit()]

global_pop = pop_df.melt(
    id_vars=pop_id_cols,
    value_vars=pop_year_cols,
    var_name='year',
    value_name='population'
)

# Renaming for consistency
global_pop = global_pop.rename(columns={
    'Country Code': 'country_code',
    'Country Name': 'country_name_pop'
})
global_pop['year'] = pd.to_numeric(global_pop['year'])

Number of unique countries after Inner Join: 195
Years available: 1970 - 2024


In [38]:
global_pop.head()

,country_name_pop,country_code,year,population
0,Aruba,ABW,1960,54922.0
1,Africa Eastern and Southern,AFE,1960,130075728.0
2,Afghanistan,AFG,1960,9035043.0
3,Africa Western and Central,AFW,1960,97630925.0
4,Angola,AGO,1960,5231654.0


#### Global Emission + Population merged data

In [33]:
# Inner Join (Keep only countries present in both datasets)
merged_global = pd.merge(
    global_em, 
    global_pop[['country_code', 'year', 'population']], 
    on=['country_code', 'year'], 
    how='inner'
)

# Clean up: Remove rows with missing values and filter for the analysis period
merged_global = merged_global.dropna(subset=['emission', 'population'])
merged_global = merged_global[merged_global['year'].between(1970, 2024)]

# Verification
print(f"Number of unique countries after Inner Join: {merged_global['country_code'].nunique()}")
print(f"Years available: {merged_global['year'].min()} - {merged_global['year'].max()}")

Number of unique countries after Inner Join: 195
Years available: 1970 - 2024


In [39]:
merged_global.head()

,country_code,country_name_em,year,emission,population
0,ABW,Aruba,1970,3.696956e+04,58950.0
1,AFG,Afghanistan,1970,1.529996e+07,11290128.0
2,AGO,Angola,1970,1.883435e+07,5852788.0
3,ALB,Albania,1970,8.230901e+06,2135479.0
4,ARE,United Arab Emirates,1970,2.977272e+07,286536.0


## Research Questions

### 1. How have greenhouse gas emissions changed across sectors, and which sector has been the most prominent contributor over the past 20 years?

In [35]:
%load_ext autoreload
%autoreload 2

from Visualization import assign_action_tiers, plot_strategic_quadrant, plot_tiered_ranking, generate_svg_line_sparkline, generate_enhanced_mfg_table
import plotly.express as px
from IPython.display import display, HTML

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
tier_metrics = assign_action_tiers(sector_em, start_year=2000, end_year=2023)

# 2. 전략적 4분면 매트릭스 (방법론 증명)
# 어떤 섹터가 왜 그 티어에 속하는지 십자선을 통해 직관적으로 보여줍니다.
plot_strategic_quadrant(tier_metrics, target_year=2023)

# 3. 티어별 수평 막대 랭킹 (최종 결과 제시)
# 19개 모든 섹터의 이름을 살리면서, 현재 배출량 규모를 티어별로 그룹핑하여 보여줍니다.
plot_tiered_ranking(tier_metrics, target_year=2023)

--- Methodology Note ---
National Avg Reduction (2000-2023): -25.39%
Pareto Threshold: Top sectors up to 85% cumulative share are 'Major'


In [37]:
mfg_html_table = generate_enhanced_mfg_table(mfg_sub_df, start_year=2000, end_year=2023, top_n=9)
display(HTML(mfg_html_table))

NameError: name 'mfg_sub_df' is not defined

Among 18 High Sectors : Electricity, Private House Holds, Manufacturing
Among manufaturing sub-sectors : Manufature of metals, non-metalic mineral products, chemical products, petroleum products. 

### 2. How responsible are Germany’s cumulative greenhouse gas emissions for the global carbon budget?

### 3. How many years are left before the global and Germany’s carbon budgets are exhausted under business-as-usual emissions?

### 4. How steep must the total emissions reduction pathway be to achieve net-zero by 2045 ?

## Conclusion and Discussion

### Summary of Findings

## References

### Primary Sources
1. 

### Cumulative world emissions data
1. 